In [19]:
# Manipulation de données
import json
import time
import sys
import io
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Appels Ollama
import os
import requests
from openai import OpenAI

from nutriscope.schema import KEY_NUTRIENTS as NUTRIENTS

In [2]:
OLLAMA_URL = os.getenv("OLLAMA_URL") or "http://localhost:11434"
MODELE_LOCAL = os.getenv("OLLAMA_MODEL") or "qwen3:8b"


def etat_ollama() -> list[str]:
    """Affiche la version d'Ollama et rend la liste des modèles installés."""
    try:
        version = requests.get(f"{OLLAMA_URL}/api/version", timeout=5).json()["version"]
        modeles = [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"]]
    except (requests.RequestException, KeyError, ValueError):
        print("Ollama ne répond pas : lancez l'application, puis relancez cette cellule.")
        return []
    print(f"Ollama {version} : {len(modeles)} modèle(s) installé(s) : {modeles}")
    return modeles


modeles_installes = etat_ollama()
OLLAMA_PRET = MODELE_LOCAL in modeles_installes
if not OLLAMA_PRET:
    print(f"Modèle {MODELE_LOCAL} indisponible : les cellules qui en ont besoin seront ignorées.")

Ollama 0.40.14 : 2 modèle(s) installé(s) : ['qwen3:4b', 'qwen3:8b']


In [3]:
client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [4]:
def ask(question, consigne=None, modele=MODELE_LOCAL, max_tokens=200):
    messages = [{"role": "user", "content": question}]
    if consigne:
        messages.insert(0, {"role": "system", "content": consigne})
    reponse = client.chat.completions.create(model=modele, messages=messages, max_tokens=max_tokens,
                                             temperature=0, reasoning_effort="none")
    return reponse.choices[0].message.content.strip()

In [56]:
df = pd.read_parquet("../data-nutriscope/nutriscope_ml_sample.parquet").set_index("code")

def find_code(code):
    try:
        return df.loc[code].to_dict()
    except KeyError:
        return {"erreur": f"aucun produit ayant pour code `{code}`"}

def get_dataframe_columns():
    return df.drop(columns=["split"]).columns.to_list()

In [48]:
def filter_dataframe_query(column: str, op: str, value):
    # Pandas gère les opérateurs classiques directement dans query()
    # On utilise '@value' pour référencer la variable Python de manière sécurisée
    query_str = f"`{column}` {op} @value"
    return df.query(query_str).to_dict()

In [ ]:
TOOLS = [
    {"type": "function",
     "function": {
         "name": "find_code",
         "description": "Donne les informations nutritionnelles d'un produit à partir de son code-barre à 8, 12 ou 13 chiffres.",
         "parameters": {"type": "object",
                        "properties": {"code": {"type": "string", "description": "Code-barre, par exemple 48218700"}},
                        "required": ["code"]}}},
    {"type": "function",
     "function": {
         "name": "filter_dataframe_query",
         "description": "Filtre les informations nutritionnelles d'un produit à partir d'une colonne.",
         "parameters": {"type": "object",
                        "properties": {
                            "column": {"type": "string", "description": "Nom de la colonne, ex: pnns_groups_1"},
                            "op": {"type": "string", "description": "Opérateur de filtre, ex: =="},
                            "value": {"type": "Any", "description": "Valeur sur lequel le filtre est exécuté, ex: Beverages"}
                        },
                        "required": ["column", "op", "value"]}}},
    {"type": "function",
     "function": {
         "name": "get_dataframe_columns",
         "description": "Donne les noms des colonnes de la table des produits.",
         "parameters": {}}},
]

ROLE = "Tu es l'assistant de l'application Nutriscope. Réponds en français, en deux phrases au plus."
PROMPT_STRUCTURE = '''Tu es l'assistant de l'application Nutriscope. Réponds en français, en deux phrases au plus.

Tâche : extraire quatre informations du message client placé entre les balises <message>.
Format : un objet JSON, et rien d'autre, avec exactement ces clés :
  "code" : le code-barre du produit, ou null
  "produit" : le produit concerné (un mot : boisson, céréales, fruit, légume), ou null
  "marque" : le nom de la marque du produit, ou null
  "nutriscore" : le nutriscore du produit, ou "non défini"
Contraintes : null quand l'information n'est pas dans le message ; n'invente rien ; pas de texte autour du JSON.'''

MAX_TOKENS = 150
TEMPERATURE = 0.0

In [ ]:
messages = [{"role": "system", "content": ROLE},
            {"role": "user", "content": "Quel est le produit ayant le code 48218700 ?"}]

answer = client.chat.completions.create(model=MODELE_LOCAL, messages=messages, tools=TOOLS,
                                         max_tokens=MAX_TOKENS, temperature=TEMPERATURE, reasoning_effort="none")
message = answer.choices[0].message

print("Texte      :", repr(message.content))
print("Demande    :", message.tool_calls[0].function)

Texte      : ''
Demande    : Function(arguments='{"code":"48218700"}', name='find_code')


In [ ]:
answer = client.chat.completions.create(model=MODELE_LOCAL, tools=TOOLS, max_tokens=MAX_TOKENS, temperature=TEMPERATURE, reasoning_effort="none",
                                         messages=[{"role": "system", "content": ROLE},
                                                   {"role": "user", "content": "Bonjour, vendez-vous des claviers sans fil ?"}])

print("Texte      :", answer.choices[0].message.content)
print("Demande    :", answer.choices[0].message.tool_calls)

Texte      : Je ne vends pas de claviers sans fil. Je peux vous aider à trouver des informations nutritionnelles sur des produits alimentaires en utilisant leur code-barre.
Demande    : None


In [74]:
FUNCTIONS = {
    "filter_dataframe_query": filter_dataframe_query,
    "find_code": find_code,
    "get_dataframe_columns": get_dataframe_columns
}

def ask_with_tools(question, outils=TOOLS, fonctions=FUNCTIONS, max_tours=3):
    messages = [{"role": "system", "content": ROLE}, {"role": "user", "content": f"<message>{question}</message>"}]
    for _ in range(max_tours):
        reponse = client.chat.completions.create(model=MODELE_LOCAL, messages=messages, tools=outils,
                                                 max_tokens=200, temperature=TEMPERATURE, reasoning_effort="none")
        message = reponse.choices[0].message
        if not message.tool_calls:                        # une réponse en texte : terminé
            return message.content
        messages.append(message)
        for appel in message.tool_calls:
            resultat = fonctions[appel.function.name](**json.loads(appel.function.arguments))
            print(f"   → {appel.function.name}({appel.function.arguments}) = {resultat}")
            messages.append({"role": "tool", "tool_call_id": appel.id, "content": json.dumps(resultat, ensure_ascii=False)})
    return "Je n'ai pas réussi à répondre, un conseiller va vous recontacter."

In [73]:
for question in ["Quel est le Nutri-Score du Nutella ?",
                 "Quelles sont les chips avec le moins de graisse ?"]:
    print(f"Client : {question}")
    print(f"Assistant : {ask_with_tools(question)}\n")

Client : Quel est le Nutri-Score du Nutella ?
Assistant : {"code": null, "produit": null, "energie": null, "graisses": null, "sucres": null, "nutriscore": "non défini"}

Client : Quelles sont les chips avec le moins de graisse ?
Assistant : {"code": null, "produit": "chips", "energie": null, "graisses": null, "sucres": null, "nutriscore": "non défini"}



In [77]:
from dataclasses import dataclass, field

def etat_ollama() -> list[str]:
    """Rend la liste des modèles installés, ou une liste vide si Ollama ne répond pas."""
    try:
        return [m["name"] for m in requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"]]
    except (requests.RequestException, KeyError, ValueError):
        return []


@dataclass
class ConfigLLM:
    fournisseur: str
    base_url: str
    api_key: str
    model: str
    prix_entree: float = 0.0   # $ par million de tokens
    prix_sortie: float = 0.0
    effort: str | None = None  # réglage du raisonnement, si le modèle en a un


def config_depuis_env() -> ConfigLLM:
    """La même configuration qu'au cours 001 : Ollama par défaut, une API si .env le demande."""
    if (os.getenv("LLM_PROVIDER") or "ollama") == "ollama":
        return ConfigLLM("ollama", f"{OLLAMA_URL}/v1", "ollama", MODELE_LOCAL, effort=os.getenv("LLM_EFFORT") or "none")
    return ConfigLLM("api", os.environ["LLM_BASE_URL"], os.environ["LLM_API_KEY"], os.environ["LLM_MODEL"],
                     float(os.getenv("LLM_PRIX_ENTREE") or 0), float(os.getenv("LLM_PRIX_SORTIE") or 0),
                     os.getenv("LLM_EFFORT") or None)


@dataclass
class Reponse:
    texte: str
    modele: str
    entree: int
    sortie: int
    cout: float
    fin: str
    latence: float = 0.0
    appels: list = field(default_factory=list)   # appels d'outils demandés par le modèle
    message: dict = field(default_factory=dict)  # le message assistant complet, à remettre dans l'historique


RESERVE_RAISONNEMENT = 2000   # tokens ajoutés à max_tokens quand le modèle raisonne (gpt-oss, qwen3 avec réflexion)
_alerte_vide = []


def alerter_reponse_vide():
    """Prévient une seule fois : une réponse vide coupée par max_tokens vient d'un modèle qui réfléchit."""
    if not _alerte_vide:
        _alerte_vide.append(True)
        print("⚠️ Réponse vide : le modèle a passé tout son budget de tokens à réfléchir. En local, utilisez qwen3:8b "
              "ou qwen3:4b-instruct (le tag qwen3:4b réfléchit toujours) ; par API, laissez LLM_EFFORT vide ou mettez low.")


def appeler(messages, cfg=None, client=None, max_tokens=512, temperature=0, outils=None, afficher=False, **options):
    """La fonction d'appel du cours 001, qui accepte désormais des outils et mesure la latence."""
    cfg = cfg or CFG
    client = client or CLIENT
    if client is None:
        raise RuntimeError("Aucun client : paquet openai absent ou modèle indisponible.")
    if cfg.effort:
        options["reasoning_effort"] = cfg.effort
    if cfg.effort != "none":
        max_tokens += RESERVE_RAISONNEMENT   # un modèle qui raisonne consomme aussi max_tokens pour réfléchir
    if outils:
        options["tools"] = outils
    debut = time.perf_counter()
    r = client.chat.completions.create(model=cfg.model, messages=messages, max_tokens=max_tokens,
                                       temperature=temperature, **options)
    latence = time.perf_counter() - debut
    m = r.choices[0].message
    appels = [{"id": a.id, "name": a.function.name, "arguments": a.function.arguments} for a in (m.tool_calls or [])]
    message = {"role": "assistant", "content": m.content or ""}
    if appels:
        message["tool_calls"] = [{"id": a["id"], "type": "function",
                                  "function": {"name": a["name"], "arguments": a["arguments"]}} for a in appels]
    cout = (r.usage.prompt_tokens * cfg.prix_entree + r.usage.completion_tokens * cfg.prix_sortie) / 1_000_000
    rep = Reponse(m.content or "", cfg.model, r.usage.prompt_tokens, r.usage.completion_tokens, cout,
                  r.choices[0].finish_reason or "", latence, appels, message)
    if afficher:
        print(f"[{rep.modele}] entrée={rep.entree} sortie={rep.sortie} latence={rep.latence:.1f}s")
        if rep.fin == "length":
            print("⚠️ Réponse coupée par max_tokens.")
    if rep.fin == "length" and not rep.texte and not appels:
        alerter_reponse_vide()
    return rep


CFG = config_depuis_env()
MODELE_PRET = OpenAI is not None and (CFG.fournisseur == "api" or CFG.model in etat_ollama())
CLIENT = OpenAI(base_url=CFG.base_url, api_key=CFG.api_key) if MODELE_PRET else None

In [80]:
EXEMPLES_EXTRACTION = [
    ("Avec quoi je peux remplacer le produit 14586955, céréales de la marque Leclerc ?",
     '{"code": "40011", "produit": "céréales", "marque": "Leclerc", "nutriscore": "non défini"}'),
    ("Le nutriscore de la pâte à tartiner Nutella est-il bien 'e' ?",
     '{"code": "40011", "produit": "pâte à tartiner", "marque": "Nutella", "nutriscore": "e"}'),
    ("Comment changer ses habitudes d'alimentation ?",
     '{"code": null, "produit": null, "marque": null, "nutriscore": "non défini"}'),
]


def messages_few_shot(ticket, exemples=EXEMPLES_EXTRACTION):
    '''La consigne structurée, puis les exemples comme de faux tours de conversation, puis le vrai ticket.'''
    messages = [{"role": "system", "content": PROMPT_STRUCTURE}]
    for question, reponse in exemples:
        messages += [{"role": "user", "content": f"<message>{question}</message>"},
                     {"role": "assistant", "content": reponse}]
    return messages + [{"role": "user", "content": f"<message>{ticket}</message>"}]

In [81]:
for question in ["Quel est le Nutri-Score du Nutella ?",
                 "Les biscuits de la marque Carrefour ne sont pas si mal en vérité",
                 "Quelles sont les chips avec le moins de graisse ?"]:
    print(f"Client : {question}")
    print(f"Assistant : {appeler(messages_few_shot(question))}\n")

Client : Quel est le Nutri-Score du Nutella ?
Assistant : Reponse(texte='{"code": "40011", "produit": "pâte à tartiner", "marque": "Nutella", "nutriscore": "e"}', modele='qwen3:8b', entree=456, sortie=38, cout=0.0, fin='stop', latence=1.2792938998900354, appels=[], message={'role': 'assistant', 'content': '{"code": "40011", "produit": "pâte à tartiner", "marque": "Nutella", "nutriscore": "e"}'})

Client : Les biscuits de la marque Carrefour ne sont pas si mal en vérité
Assistant : Reponse(texte='{"code": null, "produit": "biscuits", "marque": "Carrefour", "nutriscore": "non défini"}', modele='qwen3:8b', entree=461, sortie=33, cout=0.0, fin='stop', latence=0.6374927996657789, appels=[], message={'role': 'assistant', 'content': '{"code": null, "produit": "biscuits", "marque": "Carrefour", "nutriscore": "non défini"}'})

Client : Quelles sont les chips avec le moins de graisse ?
Assistant : Reponse(texte='{"code": null, "produit": "chips", "marque": null, "nutriscore": "non défini"}', mod